<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit2/session-06-retrieval-baseline/notebook.ipynb)


# Session 6 — A retrieval baseline

**Goal:** load a corpus with a loader that refuses bad input, index it, retrieve over it, and tell an empty result from a stale index from the wrong document.

In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [2]:
from bootcamp_agent.checks import check, review

## 1. The data shape first

A `dataclass` is a contract: named, typed fields that cross module boundaries. Retrieval scores whatever the loader hands it, so the loader is where the corpus stops being a pile of files.

In [3]:
from dataclasses import dataclass


@dataclass(frozen=True)
class MiniDocument:
    doc_id: str
    title: str
    text: str


sample = MiniDocument(doc_id="demo", title="Demo", text="Hello corpus.")
print(sample)

MiniDocument(doc_id='demo', title='Demo', text='Hello corpus.')


## 2. Exercise: a loader with two failure modes

**Context.** A loader that returns half a document on bad input is worse than one that refuses: the retrieval results that follow look completely normal and are wrong. **Errors are part of the contract.**

**Instructions.**

1. Failure mode 1 is done: a missing file raises `ValueError` naming the path.
2. Add failure mode 2: the first line must be `# <title>`; otherwise raise `ValueError` naming the file.
3. Build the `MiniDocument`: `doc_id` is the file stem, `title` is the first line without `# `, `text` is the rest.
4. Run the check. It writes a good file and a headerless file into a temp directory, and asks for a third that does not exist.

In [6]:
from pathlib import Path


def load_mini(path: Path) -> MiniDocument:
    if not path.is_file():
        raise ValueError(f"no such file: {path}")  # failure mode 1, done
    lines = path.read_text(encoding="utf-8").splitlines()
    if not lines[0].startswith("# "):
        raise ValueError(f"no title: {path}")
    return MiniDocument(doc_id=path.stem, title=lines[0][2:].strip(), text="\n".join(lines[1:]))
    # TODO(you): failure mode 2, the first line must be "#", a space, then the title
    # TODO(you): return MiniDocument(doc_id=..., title=..., text=...)
    raise NotImplementedError("add failure mode 2, then return the MiniDocument")


try:
    load_mini(Path("does-not-exist.md"))  # failure mode 1 runs today
except ValueError as error:
    print(f"refused: {error}")

refused: no such file: does-not-exist.md


**Expected output** (yours may differ in wording, not in shape):

```
refused: no such file: does-not-exist.md
✅ ch06-e2 passed
```

In [7]:
check("ch06-e2", load_mini)

✅ ch06-e2 passed


True

## 3. The real loader, and the corpus it loads

`bootcamp_agent.documents` does the same job for the teaching corpus: HTML-comment headers, a typed `CorpusError`, and a sorted glob so the load order is the same on every machine. Six documents, in git, and nothing in this session writes to them.

In [8]:
from bootcamp_agent.documents import load_corpus

documents = load_corpus(CORPUS_DIR)
for doc in documents:
    print(f"{doc.doc_id:22} {doc.title:22} tags={list(doc.tags)}")

agent-loops            Agent Loops            tags=['agent', 'loop', 'tools', 'budget', 'autonomy']
evaluation-basics      Evaluation Basics      tags=['evaluation', 'testing', 'golden-set', 'tracing', 'reliability']
mcp-overview           MCP Overview           tags=['mcp', 'tools', 'protocol', 'integration', 'agents']
prompt-injection       Prompt Injection       tags=['security', 'injection', 'untrusted-input', 'safety']
rag-basics             RAG Basics             tags=['retrieval', 'rag', 'chunking', 'citations', 'grounding']
structured-outputs     Structured Outputs     tags=['llm', 'json', 'validation', 'schema']


## 4. Exercise: index the corpus

**Context.** An index is a key and a list of documents. The corpus already ships keys, so the first index writes itself: each tag maps to the documents that carry it. Read what comes back — almost every tag points at exactly one document, which is why the next section stops using tags as keys and starts using words.

**Instructions.**

1. The loop skeleton is done. Fill the body: append `doc.doc_id` under each of its tags.
2. Sort every list, so the output is deterministic.
3. Print the index, then run the check. It rebuilds the same index from the real corpus and compares.

In [9]:
tag_index: dict[str, list[str]] = {}
for doc in documents:
    for tag in doc.tags:
        tag_index.setdefault(tag, []).append(doc.doc_id)

for ids in tag_index.values():
    ids.sort()


for tag, ids in sorted(tag_index.items()):
    print(f"{tag:16} {ids}")

agent            ['agent-loops']
agents           ['mcp-overview']
autonomy         ['agent-loops']
budget           ['agent-loops']
chunking         ['rag-basics']
citations        ['rag-basics']
evaluation       ['evaluation-basics']
golden-set       ['evaluation-basics']
grounding        ['rag-basics']
injection        ['prompt-injection']
integration      ['mcp-overview']
json             ['structured-outputs']
llm              ['structured-outputs']
loop             ['agent-loops']
mcp              ['mcp-overview']
protocol         ['mcp-overview']
rag              ['rag-basics']
reliability      ['evaluation-basics']
retrieval        ['rag-basics']
safety           ['prompt-injection']
schema           ['structured-outputs']
security         ['prompt-injection']
testing          ['evaluation-basics']
tools            ['agent-loops', 'mcp-overview']
tracing          ['evaluation-basics']
untrusted-input  ['prompt-injection']
validation       ['structured-outputs']


**Expected output** (yours may differ in wording, not in shape):

```
agent            ['agent-loops']
agents           ['mcp-overview']
...
tools            ['agent-loops', 'mcp-overview']
...
✅ ch06-e3 passed
```

In [10]:
check("ch06-e3", tag_index)

✅ ch06-e3 passed


True

## 5. Failure injection: a stale index

An index is a value computed once from a corpus that then moves on. Run this and read all four lines: a document is on disk, readable, correct — and invisible to every query. Nothing raises. Nothing is logged. That silence is the failure.

In [11]:
# \x3c is the less-than sign and \x26 the ampersand, written so GitHub's preview shows this cell.
import tempfile

scratch = Path(tempfile.mkdtemp())
(scratch / "one.md").write_text(
    "\x3c!-- title: One -->\n\x3c!-- tags: alpha -->\n\nThe first document.\n", encoding="utf-8"
)

snapshot = load_corpus(scratch)  # the index is built ONCE, from this snapshot
indexed = sorted(doc.doc_id for doc in snapshot)
tags_seen = sorted({tag for doc in snapshot for tag in doc.tags})

(scratch / "two.md").write_text(  # the corpus moves on; the index does not
    "\x3c!-- title: Two -->\n\x3c!-- tags: alpha, beta -->\n\nThe second document.\n", encoding="utf-8"
)

print("on disk :", sorted(path.stem for path in scratch.glob("*.md")))
print("indexed :", indexed)
print("tags    :", tags_seen, "->  'beta' in tags:", "beta" in tags_seen)
print("rebuilt :", sorted(doc.doc_id for doc in load_corpus(scratch)))

on disk : ['one', 'two']
indexed : ['one']
tags    : ['alpha'] ->  'beta' in tags: False
rebuilt : ['one', 'two']


## 6. Chunking: what the retriever actually sees

The retriever never sees a document. It sees chunks, each one carrying `doc_id` and `position` — the address that makes a citation checkable. Read them before you blame the model.

In [12]:
from bootcamp_agent.retrieval import chunk_document

doc = next(d for d in documents if d.doc_id == "rag-basics")
chunks = chunk_document(doc, max_chars=800)
for chunk in chunks:
    print(f"--- {chunk.doc_id}#{chunk.position} ({len(chunk.text)} chars)")
    print(chunk.text[:120].replace("\n", " "), "...")

--- rag-basics#0 (383 chars)
# RAG Basics  Retrieval-augmented generation (RAG) grounds a model's answer in documents you control. Instead of hoping  ...
--- rag-basics#1 (707 chars)
A minimal RAG pipeline has four stages. **Chunking** splits documents into passages small enough to be individually rele ...
--- rag-basics#2 (558 chars)
When a RAG system answers wrongly, the first diagnostic question is: did the right passage reach the prompt? If retrieva ...
--- rag-basics#3 (614 chars)
Every answer should name the document ids it drew from, and the application should verify those ids against what was act ...


## 7. Retrieval: inspect, always

Tokenise, keep the chunks that share a token, weight each shared token by how rare it is, sort by `(-score, doc_id, position)`. The single most useful habit in retrieval is to **look at what came back**.

In [13]:
from bootcamp_agent.retrieval import retrieve

query = "How do I tell retrieval failure from generation failure?"
for scored in retrieve(query, documents, top_k=3):
    print(f"{scored.score:6.2f}  {scored.chunk.doc_id}#{scored.chunk.position}")

  4.88  rag-basics#1
  4.88  rag-basics#2
  3.61  rag-basics#0


## 8. Exercise: the failure table

**Context.** Three queries, three different retrieval behaviours. Classifying them is the skill: a retrieval failure and a generation failure need opposite fixes, and the only way to tell them apart is to read what came back.

**Instructions.**

1. Run the cell: it prints what each query retrieved.
2. Row 1 is done as an example. Read its verdict against the output.
3. Classify rows 2 and 3: `good`, `missed` (the right doc is absent), `irrelevant` (a wrong doc is present) or `duplicated` (one document fills several slots), then say why in the same string.
4. Run the check. It re-runs retrieval and compares your verdict against what actually came back.

In [16]:
queries = [
    "what should an agent do when it cannot answer",
    "vector embeddings cosine similarity",  # note: not in our corpus vocabulary
    "citations",
]
for query in queries:
    results = retrieve(query, documents, top_k=3)
    print(f"\nQ: {query}")
    for scored in results:
        print(f"  {scored.score:6.2f}  {scored.chunk.doc_id}#{scored.chunk.position}")
    if not results:
        print("  (nothing retrieved)")

failure_table = [
    {"query": queries[0], "verdict": "good — agent-loops#2 ranked first, and it is the passage about stopping and refusing"},
    {"query": queries[1], "verdict": "missed — nothing retrivale"},  # TODO(you): verdict + why
    {"query": queries[2], "verdict": "good — rag-basics#1 ranked first, and it is the passage about stopping and refusing"},  # TODO(you): verdict + why
]


Q: what should an agent do when it cannot answer
    3.84  agent-loops#2
    2.71  prompt-injection#0
    2.71  prompt-injection#2

Q: vector embeddings cosine similarity
  (nothing retrieved)

Q: citations
    2.16  rag-basics#1
    2.16  rag-basics#2
    2.16  structured-outputs#1


**Expected output** (yours may differ in wording, not in shape):

```
Q: what should an agent do when it cannot answer
    3.84  agent-loops#2
    ...
Q: vector embeddings cosine similarity
  (nothing retrieved)
Q: citations
    2.16  rag-basics#1
    ...
✅ ch06-e1 passed
```

In [17]:
check("ch06-e1", failure_table)

✅ ch06-e1 passed


True

## 9. Failure injection: the wrong document

Two queries about chunk size, and the first sentence of every chunk that came back. The first query returns one hit with a healthy score from a document about parsing. The second puts `rag-basics` third, behind two documents that only share the word `work`. **A score measures word overlap, not relevance** — so read the chunk, not the score.

In [18]:
for query in ("what is a good chunk size", "How does chunking work?"):
    print(f"\nQ: {query}")
    for hit in retrieve(query, documents, top_k=3):
        print(f"  {hit.score:6.2f}  {hit.chunk.doc_id}#{hit.chunk.position}")
        print("         ", hit.chunk.text[:88].replace("\n", " "), "...")


Q: what is a good chunk size
    3.18  structured-outputs#2
          Parsing can fail even with a good prompt. A robust pattern is: attempt to parse; on fail ...

Q: How does chunking work?
    2.53  evaluation-basics#0
          # Evaluation Basics  "It seemed to work when I tried it" is not evidence. Evaluation rep ...
    2.53  prompt-injection#2
          No single defense is complete, but layers work. **Mark boundaries**: wrap retrieved cont ...
    2.53  rag-basics#1
          A minimal RAG pipeline has four stages. **Chunking** splits documents into passages smal ...


## 10. The full pipeline: retrieval decides first

The same agent, one question the corpus supports and one it cannot. Count the `llm_call` lines in each trace: an empty retrieval refuses **before** spending a model call, so the part capable of inventing an answer is never asked.

Read the first trace line of the supported question too. Two of its three chunks are the noise from section 9, and the answer is still cited to `rag-basics` — because `agent.py` checks the cited ids against the ids retrieval returned. Retrieval being noisy and the citation being checkable are separate properties.

In [19]:
import json

from bootcamp_agent.agent import answer_question
from bootcamp_agent.llm import FakeLLM

grounded = FakeLLM(
    responses={
        "chunking": json.dumps(
            {
                "answer": "Chunking splits documents into retrievable passages at paragraph boundaries.",
                "citations": ["rag-basics"],
                "confidence": 0.9,
                "needs_human_review": False,
            }
        )
    }
)
for query in ("How does chunking work?", "qual o placar do jogo?"):
    result = answer_question(query, documents, grounded)
    print(f"\nQ: {query}")
    for event in result.trace:
        print(f"  trace[{event.kind}] {event.detail}")
    print(f"  citations={list(result.answer.citations)} review={result.answer.needs_human_review}")


Q: How does chunking work?
  trace[retrieve] top_k=3 -> [('evaluation-basics', 0), ('prompt-injection', 2), ('rag-basics', 1)]
  trace[llm_call] attempt 1: 167 chars
  trace[decision] answered with citations ['rag-basics']
  citations=['rag-basics'] review=False

Q: qual o placar do jogo?
  trace[retrieve] top_k=3 -> []
  trace[decision] no relevant chunks; refusing without an LLM call
  citations=[] review=True


## Exit ticket

Homework: change `max_chars` (chunk size) or `top_k`, rerun the failure table, and document ONE improvement and ONE regression. There is always both.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [20]:
review("ch06")

ch06: 3/3 passed  ·  300/300 marks


True